# Bomberland PPO Training

**Session plan:**
- Phase `warmup` — 5M steps vs random/simple/smarter (~1.5 hrs on T4)
- Phase `league`  — 20M steps with self-play snapshots (~5 hrs on T4)

Checkpoints are saved to `/kaggle/working/ckpts/` and downloadable as notebook output.

**How to use:**
1. Set `PHASE` and `TOTAL_STEPS` in the Config cell.
2. Run all cells.
3. Download `ckpts/warmup_actor_final.pth` from Output tab.

In [ ]:
%%bash
pip install stable-baselines3>=2.2.1 gymnasium>=0.29.1 --quiet

In [ ]:
%%bash
set -e

# Clone participant kit (engine + baselines)
if [ ! -d /kaggle/working/kit ]; then
    git clone https://github.com/VLTisME/Bomberland-GDGoC-AI-Challenge /kaggle/working/kit --quiet
    echo "Kit cloned"
else
    echo "Kit already present"
fi

# Clone our PPO agent code
if [ ! -d /kaggle/working/project ]; then
    git clone https://github.com/PhuDoan23/GDGoC-AI-Challenge-2026 /kaggle/working/project --quiet
    echo "Project cloned"
else
    cd /kaggle/working/project && git pull --quiet
    echo "Project updated"
fi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
# Change these before running

PHASE        = "warmup"      # "warmup" or "league"
TOTAL_STEPS  = 5_000_000     # warmup=5M, league=20M
N_ENVS       = 8
N_EPOCHS     = 4             # 4 = ~2.5x faster than default 10, still good quality
N_STEPS      = 2048
BATCH_SIZE   = 64

# Set to a path if resuming from a previous run's checkpoint
LOAD_CHECKPOINT = None       # e.g. "/kaggle/working/ckpts/warmup_final"

CKPT_DIR = "/kaggle/working/ckpts"
LOG_DIR  = "/kaggle/working/logs"
# ───────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Phase: {PHASE}  |  Steps: {TOTAL_STEPS:,}  |  Device: cuda (T4)")

In [ ]:
import sys

KIT_DIR     = "/kaggle/working/kit"
PROJECT_DIR = "/kaggle/working/project"
PPO_DIR     = f"{PROJECT_DIR}/ppo_agent"

for d in (KIT_DIR, PPO_DIR, PROJECT_DIR):
    if d not in sys.path:
        sys.path.insert(0, d)

# Smoke-check imports
from engine.game import BomberEnv
from encode_obs import encode_obs
from gym_wrapper import BomberGymEnv, LeaguePool
from model import make_ppo, _best_device

device = _best_device()
print(f"Training device: {device}")

env = BomberEnv(seed=0)
obs = env.reset(seed=0)
s, a = encode_obs(obs, 0)
print(f"obs: map={s.shape}  aux={a.shape}  — engine OK")

In [ ]:
# Quick FPS benchmark before committing to full run
import time
from stable_baselines3.common.vec_env import DummyVecEnv

pool = LeaguePool(["random", "simple"])
vec_env = DummyVecEnv([lambda: BomberGymEnv(pool) for _ in range(N_ENVS)])
test_model = make_ppo(vec_env, device=device, verbose=0,
                      n_epochs=N_EPOCHS, n_steps=N_STEPS, batch_size=BATCH_SIZE)

N_WARMUP = N_STEPS * N_ENVS * 2   # 2 rollout iterations
t0 = time.time()
test_model.learn(total_timesteps=N_WARMUP)
fps = N_WARMUP / (time.time() - t0)

eta_warmup  = 5_000_000 / fps / 3600
eta_league  = 20_000_000 / fps / 3600
eta_current = TOTAL_STEPS / fps / 3600

print(f"\nFPS: {fps:.0f} steps/sec")
print(f"ETA this run ({TOTAL_STEPS/1e6:.0f}M steps): {eta_current:.1f} hrs")
print(f"ETA 5M warmup:  {eta_warmup:.1f} hrs")
print(f"ETA 20M league: {eta_league:.1f} hrs")

vec_env.close()

In [ ]:
# Build the CLI command and run training
cmd_parts = [
    f"cd {PROJECT_DIR} &&",
    f"python3 ppo_agent/train.py",
    f"--phase {PHASE}",
    f"--total_steps {TOTAL_STEPS}",
    f"--n_envs {N_ENVS}",
    f"--n_epochs {N_EPOCHS}",
    f"--n_steps {N_STEPS}",
    f"--batch_size {BATCH_SIZE}",
    f"--device {device}",
    f"--ckpt_dir {CKPT_DIR}",
    f"--log_dir {LOG_DIR}",
]
if LOAD_CHECKPOINT:
    cmd_parts.append(f"--load_checkpoint {LOAD_CHECKPOINT}")

cmd = " ".join(cmd_parts)
print("Running:\n", cmd)
os.system(cmd)

In [ ]:
# Show saved checkpoints
import os
print("\n=== Saved checkpoints ===")
for f in sorted(os.listdir(CKPT_DIR)):
    size_mb = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

In [ ]:
# Quick eval of the final actor checkpoint
import glob
actor_ckpts = glob.glob(f"{CKPT_DIR}/*actor*.pth")
if not actor_ckpts:
    print("No actor checkpoint found")
else:
    latest = sorted(actor_ckpts)[-1]
    print(f"Evaluating: {latest}\n")

    from train import eval_vs_baseline
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import DummyVecEnv

    pool2 = LeaguePool(["random", "simple"])
    vec2  = DummyVecEnv([lambda: BomberGymEnv(pool2)])
    final_model = PPO.load(f"{CKPT_DIR}/{PHASE}_final", env=vec2, device=device)

    for baseline in ["random", "simple", "smarter", "genius", "tactical"]:
        stats = eval_vs_baseline(final_model, baseline, n_games=30)
        print(f"  vs {baseline:12s}: {stats['win_rate']:5.1%}  "
              f"(W={stats['wins']} D={stats['draws']} L={stats['losses']})")
    vec2.close()

## Next steps

1. **Download checkpoint**: Go to Output tab → download `ckpts/warmup_actor_final.pth`
2. **Submit**: Run locally: `python3 scripts/pack_submission.py --checkpoint ckpts/warmup_actor_final.pth`
3. **League training**: Change `PHASE = "league"`, `TOTAL_STEPS = 20_000_000`, `LOAD_CHECKPOINT = "/kaggle/working/ckpts/warmup_final"` and re-run
